# Phase 6 — Machine Learning Models

Notebook 06. Two modeling tasks via `src/machine_learning.py`:

- **Task 1 — Rating Classification**: predict `high_rated` (rating ≥ 4.2) using
  Logistic Regression, Random Forest, and Gradient Boosting (a sklearn
  stand-in for XGBoost — swap in `XGBClassifier` for a drop-in upgrade).
- **Task 2 — Rating Regression**: predict the continuous `rating`, evaluated
  with MAE / RMSE / R², plus feature importance (SHAP if installed, else
  permutation importance).

> **Leakage guard:** `popularity_score` is deliberately excluded from the
> feature set — it's engineered as `rating * log1p(rating_count)` in Phase 1,
> so it's a direct function of the target and would leak it into both tasks.

> **Environment note:** `xgboost` and `shap` aren't installed in this sandbox
> (no network access). Gradient Boosting stands in for XGBoost, and
> permutation importance stands in for SHAP — both are fully functional and
> swap out automatically once the real packages are installed.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
%matplotlib inline


In [ ]:
from src.eda import load_features

df = load_features()
df.shape


## Task 1 — Rating Classification (high_rated ≥ 4.2)

In [ ]:
from src.machine_learning import train_classifiers, RATING_THRESHOLD

classification_results = train_classifiers(df)

pd.DataFrame({name: res["metrics"] for name, res in classification_results.items()}).T


**Result:** Random Forest is the strongest classifier (ROC-AUC ≈ 0.78,
accuracy ≈ 0.73), clearly ahead of the Logistic Regression baseline
(ROC-AUC ≈ 0.67). Gradient Boosting is close behind Random Forest. All models
struggle more with recall than precision — high-rated products that don't
"look" high-rated on these features (price/discount/text length/category
alone) are harder to catch.

## Task 2 — Rating Regression

In [ ]:
from src.machine_learning import train_regressors

regression_results = train_regressors(df)

pd.DataFrame({name: res["metrics"] for name, res in regression_results.items()}).T


**Result:** R² is low (≈0.09–0.11) across all three models — price, discount,
category, and text-length features explain only a small share of rating
variance. This is a realistic and informative negative result: it suggests
rating is driven mostly by factors *not* captured in this feature set (actual
product quality, brand reputation, individual reviewer experience), rather
than by pricing/discounting strategy. MAE ≈ 0.2–0.22 stars is nonetheless a
reasonably tight typical error given ratings cluster tightly around 4.0–4.3.

## Feature Importance

In [ ]:
from src.machine_learning import feature_importance, FEATURES

best_name = max(regression_results, key=lambda n: regression_results[n]["metrics"]["R2"])
best_pipeline = regression_results[best_name]["pipeline"]
print("Best regression model:", best_name)

X_sample = df[FEATURES].dropna().sample(300, random_state=42)
importance_df = feature_importance(best_pipeline, X_sample)
importance_df


**Interpretation:** `rating_count` and `discount_percentage` are the two
strongest predictors, followed by `discounted_price` and `main_category`. Text
length features (`review_length`, `title_length`, `description_length`)
contribute the least — consistent with the low overall R², rating is
genuinely hard to predict from these signals alone.

## Run everything in one call

Equivalent to `python src/machine_learning.py`.

In [ ]:
from src.machine_learning import run_ml

_ = run_ml(df)
